# Set up conda
```
curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash ./Miniconda3-latest-Linux-x86_64.sh

source ~/.bashrc

conda config --add channels bioconda & /
conda config --add channels conda-forge & /
conda config --set channel_priority strict & /
conda create --name bio-prak & /
conda activate bio-prak
conda instapp python=3.13 -y
conda install viennarna=2.7.2 bedtools jupyterlab scipy matplotlib pandas biopython samtools fasta -y
```

# Initial Files:
- GCF_000001405.40_GRCh38.p14_genomic.fna.gz  
    - https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.40_GRCh38.p14/GCF_000001405.40_GRCh38.p14_genomic.fna.gz
    - Full FASTA Human Genome Sequence
- GCF_000001405.40_GRCh38.p14_genomic.gtf.gz
    - https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.40_GRCh38.p14/GCF_000001405.40_GRCh38.p14_genomic.gtf.gz
    - GTF Genome Region Annotations for all Chromosomes   
- hg38.m6A.gz
    - http://bioinformaticsscience.cn/rmbase/download.php
    - Group: Mammal
    - Genome: Homo Sapiens
    - Assembly: hg38
    - Mod Type: m6A
    - Bed File with m6A Modification annotations for all Chromosomes
    - `tar xpfvz data/original/hg38.m6A.tar.gz -O | gzip -c > data/original/hg38.m6A.gz`

All saved under /data/original

# Renamed Files:
- GCF_000001405.40_GRCh38.p14_genomic.fna.gz -> genome.fna.gz
- GCF_000001405.40_GRCh38.p14_genomic.gtf.gz -> annotated.gtf.gz
- hg38.m6A.gz -> modifications.bed.gz

# Single Chromosome Files:
- Filtered with:
    ```
    zcat modifications.bed.gz | grep chr8 | gzip -c > modifications.bed.chr8.gz
    zcat annotated.gtf.gz | grep NC_000008.11 | gzip -c > annotated_NC_000008.11.gtf.gz
    zcat genome.fna.gz | sed -n '/NC_000008/,/NC_/p' | head -n -1 | gzip -c > genome_NC_000008.11.fna.gz
    ```

# Correct Chromosome Names:
```
zcat annotated_NC_000008.11.gtf.gz | sed 's/NC_000008.11/chr8/g' | gzip -c > annotated_chr8.gtf.gz
zcat genome_NC_000008.11.fna.gz | sed 's/NC_000008.11/chr8/g' | gzip -c > genome_chr8.fna.gz
```

# Bed12 Files:
```
mkdir software
cd software
git clone https://github.com/stephenfloor/extract-transcript-regions.git
```

```
mkdir data/hg38
python software/extract-transcript-regions/extract_transcript_regions.py --gtf -i data/annotated.gtf.gz -o data/hg38/hg38
```

# Bgzip Fasta for getfasta
```
zcat data/genome_chr8.fna.gz | bgzip -c > data/genome_chr8.fna.bgz
```

# GetFasta
```
bedtools getfasta -s -split -fi data/genome_chr8.fna.bgz -bed data/hg38/hg38_3utr.bed -name | gzip -c > data/hg38/fasta/hg38_3utr.fna.gz
bedtools getfasta -s -split -fi data/genome_chr8.fna.bgz -bed data/hg38/hg38_5utr.bed -name | gzip -c > data/hg38/fasta/hg38_5utr.fna.gz
bedtools getfasta -s -split -fi data/genome_chr8.fna.bgz -bed data/hg38/hg38_5utr_start.bed -name | gzip -c > data/hg38/fasta/hg38_5utr_start.fna.gz
bedtools getfasta -s -split -fi data/genome_chr8.fna.bgz -bed data/hg38/hg38_cds.bed -name | gzip -c > data/hg38/fasta/hg38_cds.fna.gz
bedtools getfasta -s -split -fi data/genome_chr8.fna.bgz -bed data/hg38/hg38_codingexons.bed -name | gzip -c > data/hg38/fasta/hg38_codingexons.fna.gz
bedtools getfasta -s -split -fi data/genome_chr8.fna.bgz -bed data/hg38/hg38_codingintrons.bed -name | gzip -c > data/hg38/fasta/hg38_codingintrons.fna.gz
bedtools getfasta -s -split -fi data/genome_chr8.fna.bgz -bed data/hg38/hg38_exons.bed -name | gzip -c > data/hg38/fasta/hg38_exons.fna.gz
bedtools getfasta -s -split -fi data/genome_chr8.fna.bgz -bed data/hg38/hg38_introns.bed -name | gzip -c > data/hg38/fasta/hg38_introns.fna.gz
bedtools getfasta -s -split -fi data/genome_chr8.fna.bgz -bed data/hg38/hg38_noncodingexons.bed -name | gzip -c > data/hg38/fasta/hg38_noncodingexons.fna.gz
bedtools getfasta -s -split -fi data/genome_chr8.fna.bgz -bed data/hg38/hg38_noncodingintrons.bed -name | gzip -c > data/hg38/fasta/hg38_noncodingintrons.fna.gz
```

# Intersects
```
intersectBed -a data/hg38/hg38_3utr.bed -b data/modifications_chr8.bed.gz -s -split | gzip -c > data/hg38/intersect/hg38_3utr_intersect.bed.gz
intersectBed -a data/hg38/hg38_5utr.bed -b data/modifications_chr8.bed.gz -s -split | gzip -c > data/hg38/intersect/hg38_5utr_intersect.bed.gz
intersectBed -a data/hg38/hg38_5utr_start.bed -b data/modifications_chr8.bed.gz -s -split | gzip -c > data/hg38/intersect/hg38_5utr_start_intersect.bed.gz
intersectBed -a data/hg38/hg38_cds.bed -b data/modifications_chr8.bed.gz -s -split | gzip -c > data/hg38/intersect/hg38_cds_intersect.bed.gz
intersectBed -a data/hg38/hg38_codingexons.bed -b data/modifications_chr8.bed.gz -s -split | gzip -c > data/hg38/intersect/hg38_codingexons_intersect.bed.gz
intersectBed -a data/hg38/hg38_codingintrons.bed -b data/modifications_chr8.bed.gz -s -split | gzip -c > data/hg38/intersect/hg38_codingintrons_intersect.bed.gz
intersectBed -a data/hg38/hg38_exons.bed -b data/modifications_chr8.bed.gz -s -split | gzip -c > data/hg38/intersect/hg38_exons_intersect.bed.gz
intersectBed -a data/hg38/hg38_introns.bed -b data/modifications_chr8.bed.gz -s -split | gzip -c > data/hg38/intersect/hg38_introns_intersect.bed.gz
intersectBed -a data/hg38/hg38_noncodingexons.bed -b data/modifications_chr8.bed.gz -s -split | gzip -c > data/hg38/intersect/hg38_noncodingexons_intersect.bed.gz
intersectBed -a data/hg38/hg38_noncodingintrons.bed -b data/modifications_chr8.bed.gz -s -split | gzip -c > data/hg38/intersect/hg38_noncodingintrons_intersect.bed.gz
```

# Make local intersects:

# Assemble Modifications